# 12b · 학습 — **ours** / transfer · seed 2,3

`ours` = acm + carry(SSCP) + BiMamba + MOSAIC.  이 창: 1모델 × 2 seed = **2잡**.

---
`transfer` = **AlohaTransferCube-v0** (`lerobot/aloha_sim_transfer_cube_human`) — insertion 보다
**짧고 쉬운** task. Intro 가 *"짧은 task 는 ACT 와 대등, 길수록 우리가 앞선다"* 고 주장하므로
**짧은 쪽 데이터포인트**가 필요하다. 프로토콜은 insertion 과 동일(150k · lr 고정 · 5rep × 500ep).
출력 경로가 task 별로 갈려 insertion 결과와 **섞이지 않는다**.


## ⚠️ 이 노트북은 **B 몫**(seed 2,3)만 돌린다 — GPU 2장짜리 노드용

| 노트북 | seed | GPU |
|---|---|---|
| **12b (이 창)** | **2, 3** | 이 노드의 GPU **0, 1** (자동) |
| 12a_train_ours_transfer_seed01.ipynb | 0, 1 | 그 노드의 GPU 0, 1 |

**GPU 는 노드마다 0번부터 자동 배정**된다 (`cf.part('B')` → 이 노드에 보이는 GPU 를 seed 수만큼).
2-GPU 노드가 두 대면 A 노드에서 `a` 를, B 노드에서 `b` 를 돌리면 된다.

> ⚠️ 없는 GPU 를 지정하면 학습이 `RuntimeError: 0 active drivers` 로 죽는다(장치가 안 보임).
> **한 노드(4-GPU)에서 a·b 를 동시에** 띄울 때만 서로 밟지 않게 직접 나눌 것:
> `SEEDS, GPUS = cf.part('B', gpus=[2, 3])`

둘 다 끝나면 seed 4개가 모여 리포트(`09`/`10`/`18`)에서 **자동으로 합쳐진다**.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK        = cf.SHORT_SIM       # 'transfer'
SEEDS, GPUS = cf.part('B')   # seed [2,3] + 이 노드의 GPU 0,1 (자동)
TAGS        = cf.GROUP_OURS

print('task :', TASK, cf.v23.TASKS[TASK])
print('seeds:', SEEDS, '| GPU:', GPUS, '| 보이는 GPU:', cf.v23.available_gpus())
print('학습 :', TAGS, '| 잡:', len(TAGS) * len(SEEDS), '|', f'{cf.STEPS:,} step')

## 커맨드 확인 — dataset/env 가 **transfer**, GPU 가 맞는지

In [ ]:
for g, s in zip(GPUS, SEEDS):
    c = cf.make_train_cmd(TAGS[0], seed=s, task=TASK, gpu_id=g)
    print(' '.join(p for p in c.split()
                   if p.startswith(('CUDA_VISIBLE_DEVICES', '--dataset.repo_id', '--env.task', '--seed'))))

## 학습 (resume 자동)
첫 실행은 transfer 데이터셋을 한 번 먼저 받는다(prefetch). 안 그러면 잡들이 같은 HF 캐시에
동시 다운로드를 걸어 대부분 죽는다.

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, gpus=GPUS)

## 상태

In [ ]:
cf.print_training_status(jobs)
print()
cf.print_ckpt_status(TAGS, SEEDS, TASK)